# Practical Session: Surface Energy Balance (SEB) & Eddy Covariance Processing

## 1. Introduction and Objectives
This notebook integrates micrometeorological observations from a surface radiation station and Eddy Covariance (EddyPro) atmospheric flux outputs. 

### Key Objectives:
1. **Time Synchronization**: Align observation timestamps by applying UTC offset adjustments ($+3\text{ hours}$).
2. **Data Regularization**: Resample multi-rate fluxes into regular $1\text{-hour}$ averages.
3. **Radiation Correction**: Correct unphysical night-time shortwave radiation readings ($S_{\uparrow}, S_{\downarrow} < 0$).
4. **Surface Energy Balance (SEB) Evaluation**:
   - Calculate Net Radiation ($R_n$):
     $$R_n = (S_{\downarrow} - S_{\uparrow}) + (L_{\downarrow} - L_{\uparrow})$$
   - Quantify total energy consumption ($H + LE + G$).
   - Compute the surface energy imbalance residual ($R_n - (H + LE + G)$).
5. **Quality Control & Outlier Filtering**: Compare raw SEB closure against outlier-filtered turbulent fluxes.

---

In [ ]:
# --- Google Colab Integration --- 
# Uncomment the following lines if you are running in Google Colab to connect your Google Drive
#from google.colab import drive
#drive.mount('/content/drive/')

#%cd /content/drive/MyDrive/cryoextremes-practicals

#!pwd

#!ls

## 2. Environment Setup
Import required standard numerical analysis, file I/O, and data visualization libraries.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 3. Data Ingestion & Time-Zone Adjustment
Load radiation and soil observations from the meteorological tower, adjust UTC timestamps by $+3\text{ hours}$, and set a unified `DatetimeIndex`.

In [ ]:
# File path to hourly meteorological tower observations
filename = "data/pross/met_data_1h.csv"

# Load data and adjust local time offset (+3 hours)
df_tower = pd.read_csv(filename)
df_tower["datetime"] = pd.to_datetime(df_tower["datetime"]) + pd.Timedelta(hours=3)
df_tower = df_tower.set_index("datetime")

df_tower

## 4. Processing Eddy Covariance (EddyPro) Outputs
Parse high-frequency flux calculation outputs, handle missing value codes (`-9999`), align time indexes, and resample parameters to $1\text{-hour}$ temporal resolution.

In [ ]:
# File path to EddyPro full output
file_path = "data/AP/eddypro_PG_OP43_full_output_2026-10-07T104038_exp.csv"

# Load output while skipping header metadata and replacing missing indicators
df = pd.read_csv(file_path, sep=",", skiprows=[0, 2], na_values=-9999)

# Select micrometeorological and state variables
cols_eddy = ["date", "time", "H", "LE", "air_temperature", "air_pressure", "RH", "wind_speed", "wind_dir"]
df_sel = df[cols_eddy].copy()

# Construct datetime index with time shift (+3 hours)
df_sel["datetime"] = pd.to_datetime(df_sel["date"] + " " + df_sel["time"]) + pd.Timedelta(hours=3)
df_sel = df_sel.set_index("datetime").drop(columns=["date", "time"])

# Restrict study period and resample to 1-hour averages
df_sel = df_sel.loc["2024-12-06":"2024-12-13"].resample("1h").mean()
df_sel

## 5. Dataset Integration & Shortwave Radiation Correction
Merge tower radiation and flux observations on common timestamps. Clean unphysical night-time shortwave radiation readings where sensors measure negative values.

In [ ]:
# Select target parameters from each source
cols_tower = ["SUp", "SDn", "LUpCo", "LDnCo", "SHF"]
cols_turbulent = ["H", "LE"]

# Merge tower observations with turbulent flux observations
df_merged = pd.merge(df_tower[cols_tower], df_sel[cols_turbulent], on="datetime", how="inner")

# Zero-out night-time shortwave radiation artifacts (SUp < 0)
df_merged.loc[df_merged['SUp'] < 0, ['SUp', 'SDn']] = 0

df_merged

## 6. Raw Surface Energy Balance (SEB) Calculation
Compute Net Radiation ($R_n$), Total Consumed Heat ($H + LE + G$), and Residual Imbalance prior to filtering flux spikes.

In [ ]:
# Compute Net Radiation (Rn) and Energy Balance components
df_merged["Rn"] = (df_merged["SDn"] + df_merged["SUp"]) + (df_merged["LDnCo"] + df_merged["LUpCo"])
df_merged["SEB_Consumed"] = df_merged["H"] + df_merged["LE"] + df_merged["SHF"]
df_merged["SEB_Residual"] = df_merged["Rn"] - df_merged["SEB_Consumed"]

df_merged

### Plot 1: Unfiltered Surface Energy Balance (SEB)
Visualization showing radiation fluxes, turbulent and soil heat fluxes, and energy balance closure using raw, unfiltered EddyPro measurements.

In [ ]:
# Figure setup for unfiltered SEB components
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# --- Subplot 1: Radiation Components ---
ax1.plot(df_merged.index, df_merged["SDn"], label="Shortwave Up ($S_{\\uparrow}$)", color="gold", linewidth=1.5)
ax1.plot(df_merged.index, df_merged["SUp"], label="Shortwave Down ($S_{\\downarrow}$)", color="orange", linestyle="--", linewidth=1.5)
ax1.plot(df_merged.index, df_merged["LDnCo"], label="Longwave Up ($L_{\\uparrow}$)", color="darkblue", linewidth=1.5)
ax1.plot(df_merged.index, df_merged["LUpCo"], label="Longwave Down ($L_{\\downarrow}$)", color="crimson", linestyle="--", linewidth=1.5)
ax1.set_ylabel("Flux ($W \cdot m^{-2}$)", fontsize=10)
ax1.set_title("Radiation Fluxes (Raw Data)", fontsize=12, fontweight="bold")
ax1.set_ylim(-700, 1000)
ax1.legend(loc="lower left", frameon=True, ncol=4)

# --- Subplot 2: Unfiltered Turbulent & Soil Heat Fluxes ---
ax2.plot(df_merged.index, df_merged["H"], label="Sensible Heat Flux ($H$)", color="tab:red", linewidth=1.5)
ax2.plot(df_merged.index, df_merged["LE"], label="Latent Heat Flux ($LE$)", color="tab:blue", linewidth=1.5)
ax2.plot(df_merged.index, df_merged["SHF"], label="Soil Heat Flux ($SHF$ / $G$)", color="tab:brown", linestyle="--", linewidth=1.5)
ax2.set_ylabel("Flux ($W \cdot m^{-2}$)", fontsize=10)
ax2.set_ylim(-200, 200)
ax2.set_title("Unfiltered Turbulent & Soil Heat Fluxes", fontsize=12, fontweight="bold")
ax2.legend(loc="lower left", frameon=True, ncol=3)

# --- Subplot 3: Unfiltered Surface Energy Balance ---
ax3.plot(df_merged.index, df_merged["Rn"], label="Net Radiation ($R_n$)", color="black", linewidth=2)
ax3.plot(df_merged.index, df_merged["SEB_Consumed"], label="Energy Consumed ($H + LE + G$)", color="darkgreen", linestyle="--", linewidth=1.5)
ax3.plot(df_merged.index, df_merged["SEB_Residual"], label="Imbalance Residual ($R_n - (H+LE+G)$)", color="purple", linestyle=":", linewidth=1.5)
ax3.set_ylabel("Flux ($W \cdot m^{-2}$)", fontsize=10)
ax3.set_xlabel("Date & Time", fontsize=11)
ax3.set_title("Surface Energy Balance (SEB) Closure (Unfiltered)", fontsize=12, fontweight="bold")
ax3.legend(loc="lower left", frameon=True, ncol=3)

plt.xticks(rotation=30)
plt.tight_layout()

# Save figure
os.makedirs("fig", exist_ok=True)
plt.savefig("fig/fig_seb_raw.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Turbulent Flux Quality Control & Outlier Filtering
Filter out spurious turbulent flux spikes exceeding realistic thresholds and linearly interpolate missing gaps across time.

In [ ]:
# Quality control filtering threshold (|Flux| > 150 W/m2)
cols_turbulent = ['H', 'LE']
df_merged[cols_turbulent] = df_merged[cols_turbulent].mask(
    (df_merged[cols_turbulent] < -150) | (df_merged[cols_turbulent] > 150)
)

# Time-based interpolation of gap values
df_merged[cols_turbulent] = df_merged[cols_turbulent].interpolate(method='time')

# Recompute energy balance terms with filtered fluxes
df_merged["SEB_Consumed"] = df_merged["H"] + df_merged["LE"] + df_merged["SHF"]
df_merged["SEB_Residual"] = df_merged["Rn"] - df_merged["SEB_Consumed"]

df_merged

### Plot 2: Quality-Controlled Surface Energy Balance (SEB)
Visualization showing radiation, turbulent, and soil fluxes after removing unrealistic spikes.

In [ ]:
# Figure setup for quality-controlled SEB components
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# --- Subplot 1: Radiation Fluxes ---
ax1.plot(df_merged.index, df_merged["SDn"], label="Shortwave Up ($S_{\\uparrow}$)", color="gold", linewidth=1.5)
ax1.plot(df_merged.index, df_merged["SUp"], label="Shortwave Down ($S_{\\downarrow}$)", color="orange", linestyle="--", linewidth=1.5)
ax1.plot(df_merged.index, df_merged["LDnCo"], label="Longwave Up ($L_{\\uparrow}$)", color="darkblue", linewidth=1.5)
ax1.plot(df_merged.index, df_merged["LUpCo"], label="Longwave Down ($L_{\\downarrow}$)", color="crimson", linestyle="--", linewidth=1.5)
ax1.set_ylabel("Flux ($W \cdot m^{-2}$)", fontsize=10)
ax1.set_title("Radiation Fluxes", fontsize=12, fontweight="bold")
ax1.set_ylim(-700, 1000)
ax1.legend(loc="lower left", frameon=True, ncol=4)

# --- Subplot 2: Quality-Controlled Turbulent & Soil Fluxes ---
ax2.plot(df_merged.index, df_merged["H"], label="Sensible Heat Flux ($H$)", color="tab:red", linewidth=1.5)
ax2.plot(df_merged.index, df_merged["LE"], label="Latent Heat Flux ($LE$)", color="tab:blue", linewidth=1.5)
ax2.plot(df_merged.index, df_merged["SHF"], label="Soil Heat Flux ($SHF$ / $G$)", color="tab:brown", linestyle="--", linewidth=1.5)
ax2.set_ylabel("Flux ($W \cdot m^{-2}$)", fontsize=10)
ax2.set_ylim(-150, 150)
ax2.set_title("Quality-Controlled Turbulent & Soil Heat Fluxes", fontsize=12, fontweight="bold")
ax2.legend(loc="lower left", frameon=True, ncol=3)

# --- Subplot 3: Quality-Controlled Surface Energy Balance ---
ax3.plot(df_merged.index, df_merged["Rn"], label="Net Radiation ($R_n$)", color="black", linewidth=2)
ax3.plot(df_merged.index, df_merged["SEB_Consumed"], label="Energy Consumed ($H + LE + G$)", color="darkgreen", linestyle="--", linewidth=1.5)
ax3.plot(df_merged.index, df_merged["SEB_Residual"], label="Imbalance Residual ($R_n - (H+LE+G)$)", color="purple", linestyle=":", linewidth=1.5)
ax3.set_ylabel("Flux ($W \cdot m^{-2}$)", fontsize=10)
ax3.set_xlabel("Date & Time", fontsize=11)
ax3.set_title("Surface Energy Balance (SEB) Closure (Filtered)", fontsize=12, fontweight="bold")
ax3.legend(loc="lower left", frameon=True, ncol=3)

plt.xticks(rotation=30)
plt.tight_layout()

# Save figure
plt.savefig("fig/fig_seb_corr.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Export Processed Micrometeorological Dataset
Combine all surface energy balance components with additional atmospheric parameters (`air_temperature`, `air_pressure`, `RH`, `wind_speed`, `wind_dir`) and save to a clean TSV file.

In [ ]:
# Concatenate additional atmospheric measurements
df_final = pd.concat([
    df_merged,
    df_sel[['air_temperature', 'air_pressure', 'RH', 'wind_speed', 'wind_dir']]
], axis=1, join="inner")

# Export dataset
output_path = "data/pross/data_tower.csv"
df_final.to_csv(output_path, sep="\t")
df_final